# SDNET2018 Crack Detection

This notebook walks through a practical image-classification workflow for SDNET2018:

1. Import libraries
2. Load the dataset from a local folder
3. Inspect the dataset structure
4. Clean and preprocess the image list
5. Explore the class balance and sample images
6. Prepare features and target labels
7. Split data into train and test sets
8. Train a baseline model
9. Evaluate performance
10. Save results and outputs

Update `DATA_DIR` in the next cell to point at your local SDNET2018 folder.

In [ ]:
from __future__ import annotations

import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["image.cmap"] = "gray"

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
LOCAL_DATA_DIR = Path("/path/to/SDNET2018")

def find_dataset_dir(input_root: Path) -> Path:
    if input_root.exists():
        candidates = []
        for path in input_root.rglob("*"):
            if path.is_dir() and any(child.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"} for child in path.rglob("*") if child.is_file()):
                candidates.append(path)
        if candidates:
            candidates.sort(key=lambda path: len(list(path.rglob("*.jpg"))) + len(list(path.rglob("*.jpeg"))) + len(list(path.rglob("*.png"))) + len(list(path.rglob("*.bmp"))), reverse=True)
            return candidates[0]
    return LOCAL_DATA_DIR

DATA_DIR = find_dataset_dir(KAGGLE_INPUT_ROOT)
print(f"Dataset directory: {DATA_DIR}")
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
SEED = 42
MODEL_NAME = "baseline_cnn"  # options: baseline_cnn, resnet18, resnet50
USE_PRETRAINED = False  # keep False in Kaggle unless internet is enabled
USE_CLASS_WEIGHTS = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load Dataset

The notebook expects SDNET2018 to be arranged in class folders such as `Cracked` and `Non-cracked`, with subfolders for bridge decks, walls, and pavements.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def infer_label(path: Path) -> int:
    parts = {part.lower() for part in path.parts}
    if "cracked" in parts or "crack" in parts:
        return 1
    if "non-cracked" in parts or "noncracked" in parts or "non_cracked" in parts:
        return 0
    raise ValueError(f"Could not infer label from path: {path}")


def infer_structure(path: Path) -> str:
    for part in path.parts:
        upper = part.upper()
        if upper in {"D", "W", "P"}:
            return upper
    return "unknown"


def infer_group_id(path: Path) -> str:
    structure = infer_structure(path)
    stem = path.stem
    prefix = stem.split("_")[0]
    prefix = re.sub(r"[^A-Za-z0-9]+", "", prefix)
    return f"{structure}_{prefix or stem}"


def discover_dataset(data_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(data_dir.rglob("*")):
        if not path.is_file() or not is_image_file(path):
            continue
        rows.append(
            {
                "path": path,
                "label": infer_label(path),
                "structure": infer_structure(path),
                "group_id": infer_group_id(path),
            }
        )
    if not rows:
        raise ValueError(f"No image files found under {data_dir}")
    return pd.DataFrame(rows)


dataset_df = discover_dataset(DATA_DIR)
dataset_df["label_name"] = dataset_df["label"].map({0: "non_cracked", 1: "cracked"})

print("Dataset shape:", dataset_df.shape)
display(dataset_df.head())
display(dataset_df[["structure", "label_name"]].value_counts().rename("count").reset_index())

## 3. Inspect Dataset Structure

We treat the folder structure as the source of labels, and we keep the file path, structure type, and group id in the DataFrame so we can avoid leakage during splitting.

In [ ]:
display(dataset_df.head(10))
print(dataset_df.dtypes)
print(dataset_df.info())
print(dataset_df["structure"].value_counts())
print(dataset_df["label_name"].value_counts())

## 4. Clean and Preprocess Data

For SDNET2018, cleaning mostly means checking for duplicate paths, missing files, and making sure the inferred labels are consistent with the folder names.

In [ ]:
dataset_df = dataset_df.drop_duplicates(subset=["path"]).copy()
dataset_df["exists"] = dataset_df["path"].apply(Path.exists)
dataset_df = dataset_df[dataset_df["exists"]].copy()

expected_labels = dataset_df["path"].apply(infer_label)
label_mismatch = dataset_df.loc[dataset_df["label"] != expected_labels]
print(f"Duplicate paths removed: {len(label_mismatch) == 0}")
print(f"Remaining rows: {len(dataset_df)}")
print(f"Label mismatches found: {len(label_mismatch)}")

dataset_df.head()

## 5. Exploratory Data Analysis

The class balance matters here because cracked images are usually the minority class, so accuracy alone is not a reliable metric.

In [ ]:
class_counts = dataset_df.groupby(["structure", "label_name"]).size().reset_index(name="count")
display(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=dataset_df, x="structure", hue="label_name", ax=axes[0])
axes[0].set_title("Counts by structure and class")
axes[0].set_xlabel("Structure")
axes[0].set_ylabel("Count")

sns.countplot(data=dataset_df, x="label_name", ax=axes[1])
axes[1].set_title("Overall class balance")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

sample_df = dataset_df.sample(min(24, len(dataset_df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
flat_axes = list(axes.flat)
for ax, (_, row) in zip(flat_axes, sample_df.iterrows()):
    image = Image.open(row["path"]).convert("RGB")
    ax.imshow(image)
    ax.set_title(f'{row["label_name"]} | {row["structure"]}', fontsize=8)
    ax.axis("off")
for ax in flat_axes[len(sample_df):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Prepare Features and Target

For this image task, the features are the image file paths loaded through a PyTorch dataset, and the target is the binary crack label.

In [ ]:
class SDNET2018Dataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), str(row["path"])


def build_transforms(image_size: int):
    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    eval_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    return train_transform, eval_transform


def group_stratified_split(frame: pd.DataFrame, train_size=0.7, val_size=0.15, test_size=0.15, seed=42):
    if not math.isclose(train_size + val_size + test_size, 1.0, abs_tol=1e-6):
        raise ValueError("train_size + val_size + test_size must sum to 1.0")

    groups = frame["group_id"].to_numpy()
    labels = frame["label"].to_numpy()
    unique_groups = np.unique(groups)

    group_labels = []
    for group in unique_groups:
        group_indices = np.where(groups == group)[0]
        group_labels.append(int(labels[group_indices].mean() >= 0.5))

    train_groups, temp_groups = train_test_split(
        unique_groups,
        test_size=(1.0 - train_size),
        random_state=seed,
        stratify=group_labels,
    )

    temp_mask = np.isin(unique_groups, temp_groups)
    temp_group_labels = [group_labels[i] for i, flag in enumerate(temp_mask) if flag]
    val_relative = val_size / (val_size + test_size)
    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=(1.0 - val_relative),
        random_state=seed,
        stratify=temp_group_labels,
    )

    train_frame = frame[frame["group_id"].isin(train_groups)].copy()
    val_frame = frame[frame["group_id"].isin(val_groups)].copy()
    test_frame = frame[frame["group_id"].isin(test_groups)].copy()
    return train_frame, val_frame, test_frame


train_frame, val_frame, test_frame = group_stratified_split(dataset_df, seed=SEED)
train_frame = train_frame.reset_index(drop=True)
val_frame = val_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

print(f"Train size: {len(train_frame)}")
print(f"Val size: {len(val_frame)}")
print(f"Test size: {len(test_frame)}")

## 7. Split Data into Train/Test Sets

We split by group id so that near-duplicate patches from the same concrete region do not leak across splits.

In [ ]:
train_transform, eval_transform = build_transforms(IMAGE_SIZE)
train_dataset = SDNET2018Dataset(train_frame, transform=train_transform)
val_dataset = SDNET2018Dataset(val_frame, transform=eval_transform)
test_dataset = SDNET2018Dataset(test_frame, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

class_weights = None
if USE_CLASS_WEIGHTS:
    class_counts = train_frame["label"].value_counts().sort_index()
    total = class_counts.sum()
    class_weights = torch.tensor([total / (2.0 * class_counts.get(i, 1)) for i in range(2)], dtype=torch.float32, device=DEVICE)

print("Train class counts:\n", train_frame["label_name"].value_counts())
print("Val class counts:\n", val_frame["label_name"].value_counts())
print("Test class counts:\n", test_frame["label_name"].value_counts())

## 8. Train a Baseline Model

Start with a simple CNN baseline, then optionally switch to a pretrained ResNet for transfer learning.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(inputs))


def build_model(model_name: str, num_classes: int = 2):
    if model_name == "baseline_cnn":
        return BaselineCNN(num_classes=num_classes)

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if USE_PRETRAINED else None
        model = models.resnet18(weights=weights)
        for parameter in model.parameters():
            parameter.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        for parameter in model.fc.parameters():
            parameter.requires_grad = True
        return model

    if model_name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if USE_PRETRAINED else None
        model = models.resnet50(weights=weights)
        for parameter in model.parameters():
            parameter.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        for parameter in model.fc.parameters():
            parameter.requires_grad = True
        return model

    raise ValueError(f"Unsupported model_name: {model_name}")


def compute_metrics(targets, predictions, probabilities):
    metrics = {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision_score(targets, predictions, zero_division=0),
        "recall": recall_score(targets, predictions, zero_division=0),
        "f1": f1_score(targets, predictions, zero_division=0),
    }
    try:
        metrics["auc"] = roc_auc_score(targets, probabilities)
    except ValueError:
        metrics["auc"] = float("nan")
    return metrics


def run_epoch(model, dataloader, optimizer, criterion, device):
    training = optimizer is not None
    model.train(training)
    all_targets = []
    all_probs = []
    all_predictions = []
    running_loss = 0.0

    for images, targets, _paths in dataloader:
        images = images.to(device)
        targets = torch.as_tensor(targets, dtype=torch.long, device=device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, targets)

        if training:
            loss.backward()
            optimizer.step()

        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = logits.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().tolist())
        all_probs.extend(probabilities.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())

    metrics = compute_metrics(all_targets, all_predictions, all_probs)
    metrics["loss"] = running_loss / max(len(dataloader.dataset), 1)
    return metrics


model = build_model(MODEL_NAME).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda parameter: parameter.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

history = []
best_state = None
best_val_loss = float("inf")

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_metrics = run_epoch(model, val_loader, None, criterion, DEVICE)
    scheduler.step(val_metrics["loss"])

    row = {f"train_{key}": value for key, value in train_metrics.items()}
    row.update({f"val_{key}": value for key, value in val_metrics.items()})
    row["epoch"] = epoch
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_f1"], label="Train")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="Val")
axes[1].set_title("F1 score")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluate Model Performance

Evaluate the best checkpoint on the held-out test set using accuracy, precision, recall, F1, AUC, and a confusion matrix.

In [ ]:
test_metrics = run_epoch(model, test_loader, None, criterion, DEVICE)
print(test_metrics)

all_targets = []
all_predictions = []
all_probabilities = []
model.eval()
with torch.no_grad():
    for images, targets, _paths in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = logits.argmax(dim=1)
        all_targets.extend(torch.as_tensor(targets).tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())

conf_matrix = confusion_matrix(all_targets, all_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.show()

metrics_summary = pd.DataFrame([test_metrics])
display(metrics_summary)

## 10. Save Results and Outputs

Save the trained model, training history, and summary metrics for later use in your report.

In [ ]:
output_dir = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("./outputs")
output_dir.mkdir(exist_ok=True)

model_path = output_dir / f"{MODEL_NAME}_sdnet2018.pt"
history_path = output_dir / f"{MODEL_NAME}_history.csv"
metrics_path = output_dir / f"{MODEL_NAME}_metrics.json"

torch.save(model.state_dict(), model_path)
history_df.to_csv(history_path, index=False)
metrics_path.write_text(pd.Series(test_metrics).to_json(indent=2))

print(f"Saved model to {model_path}")
print(f"Saved training history to {history_path}")
print(f"Saved metrics to {metrics_path}")

# Optional Grad-CAM helper for interpretation

def find_last_conv_layer(net):
    if hasattr(net, "layer4"):
        block = net.layer4[-1]
        return block.conv2 if hasattr(block, "conv2") else block
    if isinstance(net, BaselineCNN):
        return net.features[6]
    raise ValueError("Unsupported model type for Grad-CAM")


def grad_cam(net, image_tensor, target_layer, class_index=None):
    activations = None
    gradients = None

    def forward_hook(_module, _inputs, outputs):
        nonlocal activations
        activations = outputs

    def backward_hook(_module, _grad_inputs, grad_outputs):
        nonlocal gradients
        gradients = grad_outputs[0]

    handle_forward = target_layer.register_forward_hook(forward_hook)
    handle_backward = target_layer.register_full_backward_hook(backward_hook)
    try:
        net.zero_grad(set_to_none=True)
        logits = net(image_tensor)
        if class_index is None:
            class_index = int(logits.argmax(dim=1).item())
        logits[:, class_index].sum().backward()
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=image_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam
    finally:
        handle_forward.remove()
        handle_backward.remove()